In [ ]:
import json
import logging
import os
from pathlib import Path
from typing import Iterator
from dotenv import load_dotenv
import pandas as pd
import psycopg
from psycopg.rows import dict_row
from datetime import date

log = logging.getLogger(__name__)
DATA_DIR = Path("../data")


def discover_json_files(data_dir: Path) -> Iterator[Path]:
    for source in ["adm", "erp", "forecast", "tms"]:
        source_dir = data_dir / source
        if not source_dir.exists():
            log.warning("Source directory %s does not exist, skipping.", source_dir)
            continue
        yield from source_dir.rglob("*.json")

def parse_json_file(file_path: Path) -> pd.DataFrame:
    with open(file_path) as f:
        data = json.load(f)
        
    if isinstance(data, dict):
        data = [data]

    records = []
    for event in data:
        records.append({
            "event_type": file_path.stem,
            "source_system": file_path.parts[-3],
            "event_date": date.fromisoformat(file_path.parent.name),
            "payload": json.dumps(event)
        })
    df = pd.DataFrame(records)
    return df

all_dfs = []

for file_path in discover_json_files(DATA_DIR):
    log.info("Processing file: %s", file_path)
    df = parse_json_file(file_path)
    all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True)
print(combined_df.head(10))
print(combined_df["source_system"].value_counts())
print(combined_df.dtypes)


          event_type source_system  event_date  \
0       demand_plans           adm  2026-12-01   
1  inventory_targets           adm  2026-12-01   
2  inventory_targets           adm  2026-12-01   
3  inventory_targets           adm  2026-12-01   
4  inventory_targets           adm  2026-12-01   
5  inventory_targets           adm  2026-12-01   
6  inventory_targets           adm  2026-12-01   
7  inventory_targets           adm  2026-12-01   
8  inventory_targets           adm  2026-12-01   
9  inventory_targets           adm  2026-12-01   

                                             payload  
0  {"event_type": "demand_plan", "plan_id": "DP-2...  
1  {"event_type": "inventory_target", "target_id"...  
2  {"event_type": "inventory_target", "target_id"...  
3  {"event_type": "inventory_target", "target_id"...  
4  {"event_type": "inventory_target", "target_id"...  
5  {"event_type": "inventory_target", "target_id"...  
6  {"event_type": "inventory_target", "target_id"...  
7  {"even

In [ ]:
"""
CREATE TABLE IF NOT EXISTS warehouses (
    code TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    city TEXT NOT NULL,
    country_code TEXT NOT NULL,
    role TEXT NOT NULL,
    inventory_share FLOAT NOT NULL
);
"""